In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated_pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
base_path = f's3://sports-atlikon/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

print('Base Path: ', base_path)
print('Landing Path: ', landing_path)
print('Processed Path: ', processed_path)

In [0]:
bronze_table = f'{catalog}.{bronze_schema}.{data_source}'
silver_table = f'{catalog}.{silver_schema}.{data_source}'
gold_table = f'{catalog}.{gold_schema}.sb_fact_{data_source}'

In [0]:
df = (spark.read
      .options(header=True)
      .options(inferSchema=True)
      .csv(f'{landing_path}/*.csv')
      )

In [0]:
df = (
    df
    .withColumn(
        'read_timestamp',
        F.current_timestamp()
    )
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

In [0]:
print('Total Rows: ', df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write\
  .format('delta')\
  .option('delta.enableChangeDataFeed', 'true')\
  .mode('append')\
  .saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f'{processed_path}/{file_info.name}',
        True
    )

In [0]:
df_bronze = spark.sql(f'SELECT * FROM {bronze_table}')

df_bronze.show(2)

In [0]:
# 1. Keep only rows where order_qty is present
df_silver = df_bronze.filter(
    F.col('order_qty').isNotNull()
)

# 2. Clean customer id: keep numeric, else set to 999999
df_silver = (
    df_silver
    .withColumn(
        'customer_id',
        F.when(
            F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id')
        )
        .otherwise('999999')
        .cast('string')
    )
)

# 3. remove the days of the week from the order_placement_date
df_silver = df.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_silver = df_silver.withColumn(
    'order_placement_date',
    F.coalesce(
        F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
        F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
    )
)

# 5. Drop duplicates
df_silver = df_silver.dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'product_qty'])

# 6. Convert product_id to string
df_silver = df_silver.withColumn(
    'product_id',
    F.col('product_id').cast('string')
)

In [0]:
# 1. Keep only rows where order_qty is present
df_silver = df_bronze.filter(
    F.col('order_qty').isNotNull()
)

display(df_silver.select('order_qty').filter(F.col('order_qty').isNull()))

In [0]:
# 2. Clean customer id: keep numeric, else set to 999999
df_silver = (
    df_silver
    .withColumn(
        'customer_id',
        F.when(
            F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id')
        )
        .otherwise('999999')
        .cast('string')
    )
)

display(df_silver.select('customer_id').filter(F.col('customer_id').isNull()))

In [0]:
# 3. remove the days of the week from the order_placement_date
df_silver = df_silver.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

display(df_silver.filter(
    F.col("order_placement_date").rlike(r"^[A-Za-z]+,\s*")
))

In [0]:
# 4. Parse order_placement_date using multiple possible formats
df_silver = df_silver.withColumn(
    'order_placement_date',
    F.coalesce(
        F.try_to_date('order_placement_date', 'yyyy/MM/dd'),
        F.try_to_date('order_placement_date', 'dd-MM-yyyy'),
        F.try_to_date('order_placement_date', 'dd/MM/yyyy'),
        F.try_to_date('order_placement_date', 'MMMM dd, yyyy')
    )
)

In [0]:
# Check the column is now actually a DateType
print("Schema check:")
df_silver.select("order_placement_date").printSchema()

# Check how many rows failed to parse (became null after coalesce)
null_count = df_silver.filter(F.col("order_placement_date").isNull()).count()
print(f"Rows that failed to parse (null after coalesce): {null_count}")


In [0]:
# 5. Drop duplicates
df_silver = df_silver.dropDuplicates(['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])

display(
    df_silver.groupBy('order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty')
             .count()
             .filter(F.col('count') > 1)

)

In [0]:
# 6. Convert product_id to string
df_silver = df_silver.withColumn(
    'product_id',
    F.col('product_id').cast('string')
)

df_silver.select('product_id').printSchema()

In [0]:
# check the maximum and minimum date
df_silver.agg(
    F.min('order_placement_date').alias('min_date'),
    F.max('order_placement_date').alias('max_date')
).show()

In [0]:
display(df_bronze.limit(10))

In [0]:
display(df_silver.limit(10))

In [0]:
df_products = spark.table('fmcg.silver.products')

display(df_products.limit(5))

In [0]:
df_joined = df_silver.join(
    df_products,
    on='product_id',
    how='inner'
).select(df_silver['*'], df_products['product_code'])

display(df_joined.limit(10))

In [0]:
if not(spark.catalog.tableExists(silver_table)):
    df_joined.write\
        .format('delta')\
        .option('delta.enableChangeDataFeed', 'true')\
        .option('mergeSchema', 'true')\
        .mode('overwrite')\
        .saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias('silver').merge(
        df_joined.alias('bronze'),
        '''silver.order_placement_date = bronze.order_placement_date AND
           silver.order_id = bronze.order_id AND
           silver.product_code = bronze.product_code AND
           silver.customer_id = bronze.customer_id'''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## Gold

In [0]:
df_gold = spark.sql(f'''
                    SELECT order_id,
                           order_placement_date AS date,
                           customer_id AS customer_code,
                           product_code,
                           product_id,
                           order_qty AS sold_quantity
                           FROM {silver_table}
                           ''')

df_gold.show(2)

In [0]:
if not (spark.catalog.tableExists(gold_table)):
    print('creating New Table')
    df_gold.write\
        .format('delta')\
        .option('delta.enableChangeDataFeed', 'true')\
        .option('mergeSchema', 'true')\
        .mode('overwrite')\
        .saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias('source').merge(
        df_gold.alias('target'),
        '''
        source.date = target.date AND
        source.order_id = target.order_id AND
        source.customer_code = target.customer_code AND
        source.product_code = target.product_code
        source.product_id = target.product_id AND
        source.sold_quantity = target.sold_quantity'''
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
print(gold_table)

## Merge with Parent Company

In [0]:
df_child = spark.sql(f'''
                     SELECT date,
                            product_code,
                            customer_code,
                            sold_quantity
                            FROM {gold_table}''')

df_child.show()

In [0]:
# change date to first day of the month

df_monthly = (
    df_child
    # Get month start date
    .withColumn(
        'month_start',
        F.trunc('date', 'MM')
    )
    #group at monthly
    .groupBy('month_start', 'product_code', 'customer_code')
    .agg(
        F.sum('sold_quantity').alias('sold_quantity')
    )
    .withColumnRenamed(
        'month_start',
        'date'
    )
)

display(df_monthly.limit(10))

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f'{catalog}.{gold_schema}.fact_orders')
gold_parent_delta.alias('source').merge(
    df_monthly.alias('target'),
    '''
    source.date = target.date AND
    source.product_code = target.product_code AND
    source.customer_code = target.customer_code AND
    source.sold_quantity = target.sold_quantity
    '''
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()